# <center>  **<span style="font-size:80px;">AMAEM como un grafo</span>** </center>

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import pandas as pd
import numpy as np
import sys
import os 

from pathlib import Path

import osmnx as ox
import geopandas as gpd
from thefuzz import process
from shapely.ops import voronoi_diagram
from shapely.geometry import MultiPoint

sys.path.append(os.path.abspath(os.path.join("..")))
from src.config import Paths
from src.config import DatasetKeys

In [ ]:
from src.water2fraud.features.preprocessor import WaterPreprocessor
from src.water2fraud.models.water_segmenter import WaterSegmenter

preprocessor = WaterPreprocessor()

In [ ]:
seed = 80
images_path = Path("./AMAEM")
images_path.mkdir(parents=True, exist_ok=True)

# Carga de Datos

In [ ]:
df = pd.read_csv(Paths.DATA / "AMAEM.csv")

In [ ]:
df = preprocessor._process_dataframe(df)
df

# Networkx

- Comportamiento pareido
- Proximidad geográfica
- Jerarquía real de la red hidráulica

In [ ]:
barrios_csv = df[DatasetKeys.BARRIO].unique()

traduccion_osm = {
    '12-POLIGONO BABEL': 'Babel',
    '14-ENSANCHE DIPUTACION': 'Eixample',
    '16-PLA DEL BON REPOS': 'el Pla del Bon Repòs',
    '17-CAROLINAS ALTAS': 'les Carolines Altes',
    '18-CAROLINAS BAJAS': 'les Carolines Baixes',
    '25-ALTOZANO - CONDE LUMIARES': 'Altossano', 
    '33- MORANT -SAN NICOLAS BARI': 'Morant-Sant Nicolau de Bari',
    '36-CUATROCIENTAS VIVIENDAS': 'Cuatrocientas',
    '4-MERCADO': 'Centro-Mercado',
    '54-POLIGONO VALLONGA': 'Polígon industrial de la Vallonga',
    '56-DISPERSOS': 'Disseminat de l\'Alcoraya',
    'PDA VALLONGA': 'Vallonga',
    'VERDEGAS': 'el Verdegàs'
}

## Descarga

In [ ]:
print("Sincronizando con OpenStreetMap...")
gdf_osm = ox.features_from_place("Alicante, Spain", tags={
    "place": True, "landuse": True, "boundary": "administrative", "industrial": True
}).to_crs(epsg=3857)

gdf_osm['geometry'] = gdf_osm.geometry.centroid
osm_data = gdf_osm[['name', 'geometry']].dropna(subset=['name'])
nombres_osm = osm_data['name'].unique().tolist()

## Matching

In [ ]:
# Matching
mapeo_puntos = {}
for b_csv in barrios_csv:
    nombre_busqueda = traduccion_osm.get(b_csv, b_csv.split('-', 1)[-1].strip() if '-' in b_csv else b_csv)
    match_name, score = process.extractOne(nombre_busqueda, nombres_osm)
    if score >= 65:
        geom = osm_data[osm_data['name'] == match_name].geometry.iloc[0]
        mapeo_puntos[b_csv] = geom

## Report

In [ ]:
encontrados = list(mapeo_puntos.keys())
faltantes = set(barrios_csv) - set(encontrados)

print("="*40)
print(f"RESUMEN FINAL: {len(encontrados)} de {len(barrios_csv)} barrios.")
print("="*40)

## Visualización Voronoi

In [ ]:
gdf_puntos = gpd.GeoDataFrame([{'nombre_csv': k, 'geometry': v} for k, v in mapeo_puntos.items()], crs="EPSG:3857")
gdf_puntos = gdf_puntos.drop_duplicates(subset=['geometry'])
coords = [geom.coords[0] for geom in gdf_puntos.geometry]
points = MultiPoint(coords)
boundary_poly = ox.geocode_to_gdf("Alicante, Spain").to_crs(epsg=3857).geometry.iloc[0]
regiones = voronoi_diagram(points, envelope=boundary_poly)
gdf_voronoi = gpd.GeoDataFrame(geometry=[poly for poly in regiones.geoms], crs="EPSG:3857")
gdf_voronoi = gpd.clip(gdf_voronoi, boundary_poly)

gdf_grafo = gpd.sjoin(gdf_voronoi, gdf_puntos, how="inner", predicate="contains")
fig, ax = plt.subplots(figsize=(18, 18))
gdf_grafo.plot(ax=ax, column='nombre_csv', cmap='tab20', edgecolor='white', linewidth=0.5)
plt.axis('off')
plt.show()


## Coordenadas

In [ ]:
barrio_buscado = "12-POLIGONO BABEL" 
seleccion = gdf_puntos[gdf_puntos['nombre_csv'] == barrio_buscado]

if not seleccion.empty:
    punto = seleccion.geometry.iloc[0]
    
    # Coordenadas en EPSG:3857 (Metros)
    x_3857, y_3857 = punto.x, punto.y
    
    # Conversión a Latitud/Longitud (Grados decimales) para GPS
    punto_gps = seleccion.to_crs(epsg=4326).geometry.iloc[0]
    lon, lat = punto_gps.x, punto_gps.y

    print(f"Barrio: {barrio_buscado}")
    print(f"Coordenadas Proyectadas (EPSG:3857): X={x_3857:.2f}, Y={y_3857:.2f}")
    print(f"Coordenadas GPS (WGS84): Latitud={lat:.6f}, Longitud={lon:.6f}")
else:
    print(f"El barrio '{barrio_buscado}' no se encontró en el mapeo.")